# LineCount AI — Notebook 01: Conveyor Belt EDA & Detection Demo

This notebook demonstrates:
1. Simulating a conveyor belt video scenario with synthetic frames
2. Running object detection on individual frames
3. Visualising detections with bounding boxes and class labels
4. Demonstrating multi-object tracking ID assignment
5. Visualising virtual line-crossing logic

> **Dataset note:** This notebook uses synthetically generated frames (numpy + OpenCV).
> To run on real conveyor footage, replace the frame generation section with your video path.

## 1. Imports and Setup

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import cv2

sys.path.insert(0, '..')

np.random.seed(42)
print('Libraries loaded successfully.')

## 2. Synthetic Conveyor Frame Generator

In [ ]:
class SyntheticConveyor:
    """
    Generates synthetic conveyor belt frames for demonstration.
    Objects (boxes) move horizontally from right to left.
    """
    def __init__(self, width=640, height=480, n_objects=4):
        self.width = width
        self.height = height
        self.belt_y = int(height * 0.3)
        self.belt_h = int(height * 0.4)
        self.objects = [
            {
                'id': i,
                'x': width + i * 200,
                'y': self.belt_y + 20 + np.random.randint(0, self.belt_h - 80),
                'w': 60 + np.random.randint(0, 40),
                'h': 50 + np.random.randint(0, 30),
                'speed': 8 + np.random.randint(0, 5),
                'color': tuple(np.random.randint(80, 220, 3).tolist()),
                'class': np.random.choice(['box_A', 'box_B', 'box_C']),
            }
            for i in range(n_objects)
        ]

    def step(self):
        """Advance all objects by one step."""
        for obj in self.objects:
            obj['x'] -= obj['speed']
            if obj['x'] + obj['w'] < 0:
                obj['x'] = self.width + np.random.randint(50, 200)

    def render(self):
        """Render the current frame as a BGR numpy array."""
        frame = np.ones((self.height, self.width, 3), dtype=np.uint8) * 40
        # Belt
        cv2.rectangle(frame, (0, self.belt_y), (self.width, self.belt_y + self.belt_h),
                      (80, 70, 60), -1)
        # Belt lines
        for x in range(0, self.width, 60):
            cv2.line(frame, (x, self.belt_y), (x, self.belt_y + self.belt_h), (60, 55, 48), 1)
        # Objects
        for obj in self.objects:
            x, y, w, h = int(obj['x']), int(obj['y']), int(obj['w']), int(obj['h'])
            if 0 <= x < self.width or 0 <= x + w < self.width:
                cv2.rectangle(frame, (x, y), (x+w, y+h), obj['color'], -1)
                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 255, 255), 1)
                cv2.putText(frame, f"{obj['class']}#{obj['id']}",
                            (max(0, x+2), max(0, y+15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        return frame

    def get_ground_truth(self):
        """Return ground-truth bounding boxes and class labels."""
        boxes = []
        for obj in self.objects:
            x, y, w, h = int(obj['x']), int(obj['y']), int(obj['w']), int(obj['h'])
            if x < self.width and x + w > 0:
                boxes.append({'bbox': [x, y, x+w, y+h], 'class': obj['class'], 'id': obj['id']})
        return boxes


conveyor = SyntheticConveyor(n_objects=5)
print('SyntheticConveyor initialised with 5 objects.')

## 3. Visualise Synthetic Frames

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Synthetic Conveyor Belt Frames — Objects Moving Right to Left', fontsize=13)

for i, ax in enumerate(axes):
    for _ in range(15):
        conveyor.step()
    frame_rgb = cv2.cvtColor(conveyor.render(), cv2.COLOR_BGR2RGB)
    ax.imshow(frame_rgb)
    ax.set_title(f'Frame {(i+1)*15}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Simulated Detection Output

In [ ]:
# Reset conveyor for demonstration
conveyor = SyntheticConveyor(n_objects=4)
for _ in range(30):
    conveyor.step()

frame = conveyor.render()
gt_boxes = conveyor.get_ground_truth()

# Add simulated detection noise (slightly perturbed ground-truth boxes)
detections = []
for gt in gt_boxes:
    noise = np.random.randint(-5, 6, 4)
    bbox = [max(0, gt['bbox'][i] + noise[i]) for i in range(4)]
    conf = 0.75 + np.random.uniform(0, 0.2)
    detections.append({'bbox': bbox, 'class': gt['class'], 'confidence': min(1.0, conf)})

# Visualise
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
ax.imshow(frame_rgb)

colors = {'box_A': 'lime', 'box_B': 'cyan', 'box_C': 'yellow'}
for det in detections:
    x1, y1, x2, y2 = det['bbox']
    rect = mpatches.FancyBboxPatch(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2, edgecolor=colors.get(det['class'], 'white'),
        facecolor='none', boxstyle='round,pad=2'
    )
    ax.add_patch(rect)
    ax.text(x1, y1-4, f"{det['class']} {det['confidence']:.2f}",
            color=colors.get(det['class'], 'white'), fontsize=8, weight='bold')

ax.set_title('Simulated Detection Output — Bounding Boxes with Confidence Scores', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Detections: {len(detections)}')
for d in detections:
    print(f"  {d['class']:8s}  conf={d['confidence']:.3f}  bbox={d['bbox']}")

## 5. ByteTrack Assignment — Tracking IDs Across Frames

In [ ]:
from linecount.tracking.tracker import ByteTracker
from linecount.tracking.kalman import KalmanFilter

tracker = ByteTracker(max_age=15, min_hits=2, iou_threshold=0.3)

# Simulate 40 frames of tracking
conveyor = SyntheticConveyor(n_objects=4)
track_history = {}

for frame_idx in range(40):
    conveyor.step()
    gt_boxes = conveyor.get_ground_truth()
    # Simulate detections with slight noise
    sim_detections = []
    for gt in gt_boxes:
        noise = np.random.randint(-3, 4, 4)
        bbox = [max(0, gt['bbox'][i] + noise[i]) for i in range(4)]
        sim_detections.append({'bbox': bbox, 'conf': 0.8, 'class_id': 0, 'class_name': gt['class']})
    tracks = tracker.update(sim_detections)
    for track in tracks:
        tid = track.get('track_id', -1)
        if tid not in track_history:
            track_history[tid] = []
        bbox = track.get('bbox', [0,0,0,0])
        cx = (bbox[0] + bbox[2]) / 2
        cy = (bbox[1] + bbox[3]) / 2
        track_history[tid].append((frame_idx, cx, cy))

# Plot trajectories
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 640)
ax.set_ylim(0, 480)
ax.invert_yaxis()
ax.set_facecolor('#1a1a2e')

cmap = plt.cm.Set1
for i, (tid, history) in enumerate(track_history.items()):
    xs = [h[1] for h in history]
    ys = [h[2] for h in history]
    color = cmap(i % 9)
    ax.plot(xs, ys, '-o', markersize=3, color=color, linewidth=1.5, label=f'Track #{tid}')
    if xs:
        ax.annotate(f'#{tid}', (xs[-1], ys[-1]), color=color, fontsize=9, weight='bold')

# Virtual line
line_x = 300
ax.axvline(x=line_x, color='red', linewidth=2, linestyle='--', label='Counting line')

ax.set_title('Object Trajectories with Persistent Track IDs (40 frames)', fontsize=12, color='white')
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('white')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlabel('X position (pixels)', color='white')
ax.set_ylabel('Y position (pixels)', color='white')
plt.tight_layout()
plt.show()

print(f'Total unique tracks created: {len(track_history)}')

## 6. Line-Crossing Logic Visualisation

In [ ]:
from linecount.counting.counter import LineCrossingCounter

# Counting line at x=300 (horizontal line)
counter = LineCrossingCounter(line_start=(300, 100), line_end=(300, 380))

conveyor2 = SyntheticConveyor(n_objects=3)
tracker2 = ByteTracker(max_age=15, min_hits=2, iou_threshold=0.3)

crossing_events = []
frame_counts = []

for frame_idx in range(120):
    conveyor2.step()
    gt_boxes = conveyor2.get_ground_truth()
    sim_dets = [
        {'bbox': gt['bbox'], 'conf': 0.85, 'class_id': 0, 'class_name': gt['class']}
        for gt in gt_boxes
    ]
    tracks = tracker2.update(sim_dets)
    before = counter.count if hasattr(counter, 'count') else 0
    counter.update(tracks)
    after = counter.count if hasattr(counter, 'count') else 0
    if after > before:
        crossing_events.append(frame_idx)
    frame_counts.append(after)

# Plot cumulative count
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(120), frame_counts, color='#00d4ff', linewidth=2)
for ev in crossing_events:
    ax.axvline(x=ev, color='orange', alpha=0.7, linewidth=1, linestyle='--')
ax.set_title('Cumulative Object Count vs Frame Number', fontsize=12)
ax.set_xlabel('Frame')
ax.set_ylabel('Total objects counted')
ax.yaxis.get_major_locator().set_params(integer=True)
ax.grid(alpha=0.3)
handles = [
    Line2D([0], [0], color='#00d4ff', linewidth=2, label='Cumulative count'),
    Line2D([0], [0], color='orange', linewidth=1, linestyle='--', label='Crossing event'),
]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()

print(f'Total crossings detected: {frame_counts[-1] if frame_counts else 0}')
print(f'Crossing events at frames: {crossing_events}')

## 7. Kalman Filter State Evolution

In [ ]:
# Demonstrate Kalman filter prediction during a brief occlusion
kf = KalmanFilter()

# Simulate an object moving horizontally at constant speed
true_positions = [(100 + i*8, 200) for i in range(30)]

# Initialise KF with first measurement
kf.initiate(np.array([100, 200, 60, 50], dtype=float))  # x,y,w,h

predicted_xs = []
observed_xs = []

for i, (tx, ty) in enumerate(true_positions):
    mean, cov = kf.predict()
    predicted_xs.append(mean[0])
    # Simulate occlusion between frames 10-15
    if 10 <= i <= 15:
        observed_xs.append(None)  # no measurement
    else:
        noise = np.random.randn() * 3
        measurement = np.array([tx + noise, ty + noise, 60, 50], dtype=float)
        mean, cov = kf.update(mean, cov, measurement)
        kf.mean = mean
        kf.covariance = cov
        observed_xs.append(tx)

fig, ax = plt.subplots(figsize=(12, 4))
frames = list(range(30))
ax.plot(frames, [p[0] for p in true_positions], 'g-', label='Ground truth X', linewidth=2)
ax.plot(frames, predicted_xs, 'b--', label='Kalman prediction X', linewidth=2)
obs = [(f, x) for f, x in zip(frames, observed_xs) if x is not None]
occ = [(f, predicted_xs[f]) for f, x in zip(frames, observed_xs) if x is None]
if obs:
    ax.scatter([o[0] for o in obs], [o[1] for o in obs], c='lime', zorder=5, s=30, label='Observations')
if occ:
    ax.scatter([o[0] for o in occ], [o[1] for o in occ], c='red', marker='x', zorder=5, s=60, label='Occluded (predict only)')
ax.axvspan(10, 15, alpha=0.15, color='red', label='Occlusion window')
ax.set_title('Kalman Filter: X Position During Occlusion (frames 10–15)', fontsize=12)
ax.set_xlabel('Frame')
ax.set_ylabel('X position (pixels)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('The Kalman filter smoothly predicts position during occlusion.')
print('When measurements resume, it re-converges to the true trajectory.')

## Summary

This notebook demonstrated:

| Step | Component | Output |
|------|-----------|--------|
| Frame generation | `SyntheticConveyor` | BGR frames with moving boxes |
| Object detection | Simulated bounding boxes | Detections with confidence scores |
| Multi-object tracking | `ByteTracker` | Persistent track IDs |
| Line crossing | `LineCrossingCounter` | Cumulative count + crossing events |
| Occlusion handling | `KalmanFilter` | Smooth trajectory prediction |

**Next step:** Run notebook 02 for tracking benchmark analysis across different IoU thresholds.